# Verb frame against predicament

One panel per model over the `severity_verb` dataset: 20 predicaments crossed with 4 verb
frames, each written in 2 moods, 160 rows per model.

The objection this answers is that the task ordering in the wording corpus looks like a
property of the verb rather than of the situation - that "find my house keys" and "get back
into my locked house" land at opposite ends while naming much the same trouble. In that
corpus the objection cannot be settled, because each task has exactly one phrasing and so
task identity and verb frame are the same variable wearing two names.

`scripts/corpora/inference/severity_verb_prompts.py` separates them by construction. Each
predicament is fixed by a situation sentence copied verbatim into every one of its prompts,
and the request that follows names the missing object only by pronoun, so the verb carries
no information about what went missing. Every predicament appears under every frame, and
every frame is written both as an instruction and as a question. That mood is the
replicate: it is what lets the predicament-by-frame interaction be tested against real
error rather than standing in for it, and it means a frame effect has to survive being said
two ways before it counts as an effect of the verb.

## Configuration

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import f as f_distribution, spearmanr
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'scripts' / 'stakes_surface_pipeline.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.corpora.inference.severity_verb_prompts import (
    MOODS, PREDICAMENTS, VERB_FRAMES)

ARTIFACT_ROOT = ROOT / 'artifacts' / 'content' / 'artifacts'
MODELS = None                          # None -> every model directory with the dataset

DATASET = 'severity_verb'
ARC_CSV = f'{DATASET}_arc_lengths.csv'
RATINGS_CSV = 'stated_stakes_summary.csv'   # optional; written by the rating pass

TARGET = 'arc_length_parallel'
PREDICAMENT, FRAME, MOOD, PRONOUN = 'predicament', 'verb_frame', 'mood', 'object_pronoun'

# The corpus is the source of truth for a complete design, so a half-written CSV
# fails here rather than quietly producing an unbalanced decomposition.
PREDICAMENT_ORDER = [predicament[0] for predicament in PREDICAMENTS]
FRAME_ORDER = [frame[0] for frame in VERB_FRAMES]
PRONOUN_OF = {predicament[0]: predicament[1] for predicament in PREDICAMENTS}

SURFACE, GRID = '#ffffff', '#e6e5e1'
INK_PRIMARY, INK_SECONDARY, INK_MUTED = '#0b0b0b', '#52514e', '#8a8983'
# Four frames with no natural order between them, so categorical hues, not a ramp.
FRAME_COLOURS = {'locate': '#2a78d6', 'recover': '#c2571a',
                 'retrieve': '#1f8a70', 'resolve': '#8a5cd6'}
COLUMNS = 3

print(f'Repository root: {ROOT}')
print(f'Design:          {len(PREDICAMENT_ORDER)} predicaments x {len(FRAME_ORDER)} frames '
      f'x {len(MOODS)} moods = {len(PREDICAMENT_ORDER) * len(FRAME_ORDER) * len(MOODS)} rows')
print(f'Frames:          {", ".join(FRAME_ORDER)}')
print(f'Moods:           {", ".join(MOODS)}')

## Loading, and what counts as a usable table

Every model directory carrying the dataset is read. The design must be complete and
balanced before any of the arithmetic below means anything - one row per predicament, frame
and mood, nothing missing, nothing doubled - so that is checked rather than assumed. A model
whose inference pass has not run yet is skipped with a note, not silently dropped.

In [ ]:
def load_model(directory):
    """One model's 160 rows as a tidy frame, or None if the pass has not run."""
    path = directory / 'inference' / DATASET / ARC_CSV
    if not path.is_file():
        return None
    rows = pd.read_csv(path)
    missing = {PREDICAMENT, FRAME, MOOD, TARGET} - set(rows.columns)
    if missing:
        raise ValueError(f'{path}: missing columns {sorted(missing)}')
    if set(rows[PREDICAMENT]) != set(PREDICAMENT_ORDER):
        raise ValueError(f'{path}: predicaments do not match the corpus.')
    if set(rows[FRAME]) != set(FRAME_ORDER) or set(rows[MOOD]) != set(MOODS):
        raise ValueError(f'{path}: frames or moods do not match the corpus.')
    if not rows.groupby([PREDICAMENT, FRAME, MOOD]).size().eq(1).all():
        raise ValueError(f'{path}: the design is not balanced with one row per cell.')
    rows[PRONOUN] = rows[PREDICAMENT].map(PRONOUN_OF)
    return rows


directories = ([path for path in sorted(ARTIFACT_ROOT.iterdir()) if path.is_dir()]
               if MODELS is None else [ARTIFACT_ROOT / name for name in MODELS])
tables, skipped = {}, []
for directory in directories:
    rows = load_model(directory)
    (skipped.append(directory.name) if rows is None
     else tables.__setitem__(directory.name, rows))

if skipped:
    print(f'No {ARC_CSV} yet for: {", ".join(skipped)}')
if not tables:
    raise ValueError(f'No model has {ARC_CSV}; run severity_verb_inference.run first.')
print(f'{len(tables)} model(s) loaded: {", ".join(tables)}')

## The decomposition

The design is complete and balanced, so the spread in the coordinate splits into orthogonal
pieces that sum exactly:

```
SS_total = SS_predicament + SS_frame + SS_mood + SS_predicament*frame + SS_residual
  df 159       df 19          df 3       df 1          df 57              df 79
```

`eta_squared` is each term's share of the total, and the first two columns are what the
objection is actually about: if the verb frame drove the ordering, its share would be the
large one.

The mood replicate is what makes the rest of the table honest. Without it the
predicament-by-frame interaction and the error term are the same numbers, so additivity has
to be assumed rather than checked; here the interaction is tested against a residual it does
not contain. A large interaction with small main effects means the two factors are not
separable in the way the table presumes, and the panels should be read before anything is
claimed.

`frame_range_share` sits beside the F ratios on purpose. With 3 degrees of freedom against
79 the frame term is well powered to detect a small effect, and an effect can be entirely
real and still be trivial next to the predicament one.

In [ ]:
EFFECTS = [('predicament', [PREDICAMENT]), ('frame', [FRAME]), ('mood', [MOOD]),
           ('predicament*frame', [PREDICAMENT, FRAME])]


def decompose(rows):
    """Balanced ANOVA over predicament, frame, mood and the two-way interaction."""
    values = rows[TARGET].to_numpy(dtype=np.float64)
    grand = values.mean()
    ss_total = ((values - grand) ** 2).sum()

    sums, degrees = {}, {}
    for name, keys in EFFECTS:
        # A balanced design makes each effect the cell means minus everything already
        # accounted for below it, so the pieces are orthogonal and sum to the total.
        cell = rows.groupby(keys, observed=True)[TARGET].agg(['mean', 'size'])
        raw = (cell['size'] * (cell['mean'] - grand) ** 2).sum()
        raw_df = len(cell) - 1
        for lower, lower_keys in EFFECTS:
            if set(lower_keys) < set(keys):
                raw -= sums[lower]
                raw_df -= degrees[lower]
        sums[name], degrees[name] = raw, raw_df

    ss_residual = ss_total - sum(sums.values())
    df_residual = len(values) - 1 - sum(degrees.values())
    ms_residual = ss_residual / df_residual

    result = {}
    for name, _ in EFFECTS:
        statistic = (sums[name] / degrees[name]) / ms_residual
        result[f'{name} eta^2'] = sums[name] / ss_total
        result[f'F {name}'] = statistic
        result[f'p {name}'] = float(f_distribution.sf(statistic, degrees[name], df_residual))
    result['residual eta^2'] = ss_residual / ss_total

    predicament_means = rows.groupby(PREDICAMENT, observed=True)[TARGET].mean()
    frame_means = rows.groupby(FRAME, observed=True)[TARGET].mean()
    result['predicament range'] = np.ptp(predicament_means)
    result['frame range'] = np.ptp(frame_means)
    result['frame_range_share'] = np.ptp(frame_means) / np.ptp(predicament_means)
    return result


decomposition = pd.DataFrame({name: decompose(rows) for name, rows in tables.items()}).T
decomposition.index.name = 'model'
shares = [column for column in decomposition.columns if column.endswith('eta^2')]
display(decomposition[shares + ['predicament range', 'frame range',
                                'frame_range_share']].round(4))
display(decomposition[[c for c in decomposition.columns
                       if c.startswith(('F ', 'p '))]].round(4))

for name in decomposition.index:
    row = decomposition.loc[name]
    print(f'{name:24s} predicament {100 * row["predicament eta^2"]:5.1f}%  '
          f'frame {100 * row["frame eta^2"]:5.1f}%  mood {100 * row["mood eta^2"]:4.1f}%  '
          f'interaction {100 * row["predicament*frame eta^2"]:5.1f}%  '
          f'(frame moves {row["frame_range_share"]:.2f}x what the predicament does)')

### Does the frame effect survive being said twice?

Each frame's average departure from its model's grand mean, computed separately inside each
mood. A verb effect should not care how the request was phrased, so the two columns for a
frame should agree; where they do not, what looked like a verb effect is a property of one
sentence.

`length bound` is the most the frame effect could owe to prompt length. The four frames
differ by at most one token, and the within-class slope of the coordinate on token length
measured elsewhere in this repository is under 0.15 per token, so anything above this bound
is not a length artifact.

In [ ]:
LENGTH_SLOPE_BOUND = 0.15      # arc units per token; see the matched-length control
FRAME_TOKEN_SPREAD = 1.0       # the four frames differ by at most one token

effects = {}
for name, rows in tables.items():
    grand = rows[TARGET].mean()
    by_mood = rows.groupby([MOOD, FRAME], observed=True)[TARGET].mean() - grand
    effects[name] = {f'{frame} ({mood})': by_mood.loc[(mood, frame)]
                     for frame in FRAME_ORDER for mood in MOODS}
effects = pd.DataFrame(effects).T
effects.index.name = 'model'
display(effects.round(3))

consistency = {}
for name, rows in tables.items():
    wide = rows.groupby([FRAME, MOOD], observed=True)[TARGET].mean().unstack()
    disagreement = (wide[MOODS[0]] - wide[MOODS[1]]).abs().max()
    consistency[name] = {
        'frame effects agree across mood within': disagreement,
        'frame range': decomposition.loc[name, 'frame range'],
        'length bound': LENGTH_SLOPE_BOUND * FRAME_TOKEN_SPREAD,
        'frame range beats length bound': bool(
            decomposition.loc[name, 'frame range'] > LENGTH_SLOPE_BOUND * FRAME_TOKEN_SPREAD),
    }
display(pd.DataFrame(consistency).T)

### Is it the pronoun rather than the predicament?

The object-directed frames name the missing thing as `it` or `them`, and that word is fixed
by the predicament, so a number-agreement effect would arrive dressed as a predicament
effect. The corpus keeps the two groups close to balanced precisely so this can be checked.
The comparison is made inside the object-directed frames, where the pronoun actually appears,
and against `resolve`, where it does not - a pronoun effect should show up in the first and
not the second.

In [ ]:
pronoun_check = {}
for name, rows in tables.items():
    directed = rows[rows[FRAME] != 'resolve']
    undirected = rows[rows[FRAME] == 'resolve']
    gap = (directed[directed[PRONOUN] == 'them'][TARGET].mean()
           - directed[directed[PRONOUN] == 'it'][TARGET].mean())
    placebo = (undirected[undirected[PRONOUN] == 'them'][TARGET].mean()
               - undirected[undirected[PRONOUN] == 'it'][TARGET].mean())
    pronoun_check[name] = {
        'them - it, object-directed frames': gap,
        'them - it, resolve frame (placebo)': placebo,
        'difference': gap - placebo,
        'predicament range': decomposition.loc[name, 'predicament range'],
    }
display(pd.DataFrame(pronoun_check).T.round(3))
print('A pronoun effect shows as a nonzero difference between the first two columns. '
      'Both columns moving together is a predicament imbalance, not a pronoun effect.')

## The panels

Predicaments along the x axis, ordered by that model's own mean over frames and moods, with
one line per frame (averaged over mood). If the four lines run close together and roughly
parallel, the verb is a small additive shift and the situation is doing the work. If they
cross and separate, the objection is right.

In [ ]:
names = list(tables)
row_count = int(np.ceil(len(names) / COLUMNS))
figure = make_subplots(rows=row_count, cols=COLUMNS, subplot_titles=names,
                       horizontal_spacing=0.075, vertical_spacing=0.18)

for index, name in enumerate(names):
    row, column = divmod(index, COLUMNS)
    wide = tables[name].groupby([PREDICAMENT, FRAME], observed=True)[TARGET].mean().unstack()
    wide = wide.loc[wide.mean(axis=1).sort_values(kind='stable').index, FRAME_ORDER]
    positions = np.arange(len(wide))
    for frame in FRAME_ORDER:
        figure.add_trace(go.Scatter(
            x=positions, y=wide[frame].to_numpy(), mode='lines+markers',
            name=frame, legendgroup=frame, showlegend=index == 0,
            line=dict(color=FRAME_COLOURS[frame], width=1.6),
            marker=dict(color=FRAME_COLOURS[frame], size=6,
                        line=dict(color=SURFACE, width=1)),
            customdata=np.array(wide.index),
            hovertemplate=f'{frame}<br>%{{customdata}}<br>%{{y:.2f}}<extra></extra>'),
            row=row + 1, col=column + 1)
    figure.update_xaxes(tickmode='array', tickvals=positions, ticktext=list(wide.index),
                        tickangle=-60, row=row + 1, col=column + 1)
    if column == 0:
        figure.update_yaxes(title_text='arc_length_parallel', row=row + 1, col=column + 1)

for index in range(len(names), row_count * COLUMNS):
    row, column = divmod(index, COLUMNS)
    figure.update_xaxes(visible=False, row=row + 1, col=column + 1)
    figure.update_yaxes(visible=False, row=row + 1, col=column + 1)

figure.update_xaxes(gridcolor=GRID, zeroline=False, linecolor=GRID,
                    tickfont=dict(size=8, color=INK_SECONDARY))
figure.update_yaxes(gridcolor=GRID, zeroline=False, linecolor=GRID,
                    title_font=dict(size=11, color=INK_SECONDARY),
                    tickfont=dict(size=10, color=INK_SECONDARY))
for annotation in figure.layout.annotations[:len(names)]:
    annotation.font.update(size=12, color=INK_PRIMARY)
figure.add_annotation(
    text='Predicaments are ordered by each model\'s own mean, so the x axis differs between '
         'panels; each point averages the two moods. Four lines close and parallel means the '
         'verb is a shift, not the ordering.',
    xref='paper', yref='paper', x=0, y=1.045, xanchor='left', yanchor='bottom',
    showarrow=False, align='left', font=dict(size=11, color=INK_SECONDARY))
figure.update_layout(template='plotly_white', height=420 * row_count,
                     title=dict(text='Verb frame against predicament',
                                x=0, xanchor='left', font=dict(size=16, color=INK_PRIMARY)),
                     paper_bgcolor=SURFACE, plot_bgcolor=SURFACE,
                     margin=dict(t=140, l=75, r=30, b=110),
                     legend=dict(orientation='h', y=1.02, x=1, xanchor='right',
                                 yanchor='bottom', font=dict(size=11, color=INK_SECONDARY)),
                     hoverlabel=dict(bgcolor=SURFACE, bordercolor=GRID,
                                     font=dict(size=11, color=INK_PRIMARY)))
figure.show()

## Does the ordering hold inside every frame?

The sharpest form of the objection is that whichever end is high, the ordering is not about
the situation. Holding the frame fixed answers it directly: within one frame the request is
identical word for word bar the pronoun, so the predicament ordering there owes nothing to
the verb. If the four orderings agree with each other, there is a predicament ordering to
talk about at all.

`vs stated stakes` compares that ordering against the model's own ratings of the same
prompts, which the rating pass writes to `stated_stakes_summary.csv`. It is skipped when that
pass has not been run - it is a separate generation step, not part of the projection.

In [ ]:
agreement = {}
for name, rows in tables.items():
    wide = rows.groupby([PREDICAMENT, FRAME], observed=True)[TARGET].mean().unstack()
    correlations = [spearmanr(wide[first], wide[second])[0]
                    for i, first in enumerate(FRAME_ORDER)
                    for second in FRAME_ORDER[i + 1:]]
    agreement[name] = {'lowest frame-pair rho': min(correlations),
                       'median frame-pair rho': float(np.median(correlations)),
                       'highest frame-pair rho': max(correlations)}
print('Agreement between the four within-frame predicament orderings')
display(pd.DataFrame(agreement).T.round(3))


def stated_stakes(directory):
    """Mean stated rating per predicament, or None if the rating pass has not run."""
    path = directory / 'inference' / DATASET / RATINGS_CSV
    if not path.is_file():
        return None
    summary = pd.read_csv(path)
    return summary.groupby('template_id').rating_median.mean().reindex(PREDICAMENT_ORDER)


rated_rows = []
for name in names:
    rated = stated_stakes(ARTIFACT_ROOT / name)
    if rated is None or rated.isna().any():
        continue
    wide = tables[name].groupby([PREDICAMENT, FRAME], observed=True)[TARGET].mean().unstack()
    for frame in [*FRAME_ORDER, 'all frames']:
        column = wide.mean(axis=1) if frame == 'all frames' else wide[frame]
        rho, p = spearmanr(column.reindex(PREDICAMENT_ORDER), rated)
        rated_rows.append({'model': name, 'frame': frame,
                           'rho vs stated stakes': round(rho, 3), 'p': round(p, 4)})
if rated_rows:
    display(pd.DataFrame(rated_rows).set_index(['model', 'frame']))
else:
    print(f"No {RATINGS_CSV} for this dataset yet; add {DATASET!r} to RATING_DATASETS "
          f"in notebooks/arc_length_cache.ipynb to fill this in.")

## Reading the result

The objection is answered by the first table, not by any of the pictures:

- **`frame eta^2` small beside `predicament eta^2`**, with the within-frame orderings agreeing
  and the frame effects steady across mood - the coordinate is responding to the situation,
  and the wording corpus's task ordering was confounded but not wrong.
- **`frame eta^2` comparable to or larger than `predicament eta^2`** - the objection holds, and
  any task ordering taken from the wording corpus has to be withdrawn rather than defended.
- **`predicament*frame eta^2` large with both main effects small** - the two factors are not
  additive here and neither summary means much on its own; read the panels first.
- **frame effects that disagree between moods** - whatever moved is a property of a sentence
  rather than of a verb, and the frame labels are not measuring what they claim.

Nothing here asserts which of those is the case. It is a measurement, and the corpus was
built so that the measurement could be made at all.